# 4.4 — Aggregation and Join Types

**Chapter 4, sections 4.3.4 and 4.3.5.**

**The question this notebook answers:** both halves of this notebook are about spending a
shuffle. Aggregation is where distributed cost concentrates, and the question is how to spend
one shuffle well rather than four badly. A join is the other shuffle, and there the question is
not how it executes — the optimizer decides that — but **which rows survive**, which no
optimizer decides.

The mechanics of joins were covered by chapter 3's
[3.5](03.05%20DataFrame%20Joins.ipynb). What is new here is the policy and the three standard
failures.

**Data.** The taxi file, two million rows, plus a vehicle reference table built from it — with
**four fifths of the medallions**, so that a fifth of the trips have no reference row and the
choice of join type has visible consequences.

Runs on a laptop in about a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, time, logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-4.4")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)
spark.conf.set("spark.sql.session.timeZone", "UTC")     # set deliberately; see 4.3

names = ["medallion", "hack_license", "pickup_datetime", "dropoff_datetime",
         "trip_time", "trip_distance", "pickup_longitude", "pickup_latitude",
         "dropoff_longitude", "dropoff_latitude", "payment_type", "fare_amount",
         "surcharge", "mta_tax", "tip_amount", "tolls_amount", "total_amount"]
types = {"medallion": StringType(), "hack_license": StringType(),
         "pickup_datetime": TimestampType(), "dropoff_datetime": TimestampType(),
         "trip_time": IntegerType(), "payment_type": StringType()}
schema = StructType([StructField(n, types.get(n, DoubleType()), True) for n in names])

df = (spark.read.schema(schema).option("header", "false")
      .csv(f"{DATA}/taxi-data-sorted-small.csv.bz2")
      .drop("pickup_longitude", "pickup_latitude",
            "dropoff_longitude", "dropoff_latitude")
      # the fare band of the course notebook, used as a grouping key throughout
      .withColumn("fare_category", F.col("fare_amount").cast("int"))
      .cache())
print(f"{df.count():,} trips | Spark {spark.version}")

1,999,999 trips | Spark 4.2.0


## An instrument: counting jobs

Most of the claims in this section are about *how much work* a query costs, not about its
answer, so we need an instrument that sees the work. Two are used below.

* **Jobs.** Every action submits one or more Spark jobs. `statusTracker` reports how many
  landed in a named job group, and that count is what an extra pass over the data looks like
  from outside.
* **Exchanges.** One `Exchange` node in the physical plan is one shuffle. Counting them in the
  plan is exact and needs no execution at all.

In [2]:
def measure(label, fn, quiet=False):
    """Run fn as its own job group; report the number of jobs and the wall clock."""
    group = f"cs777-{time.time_ns()}"
    sc.setJobGroup(group, label)
    started = time.time()
    result = fn()
    elapsed = time.time() - started
    jobs = len(sc.statusTracker().getJobIdsForGroup(group))
    sc.setJobGroup(None, None)
    if not quiet:
        print(f"  {label:44s} jobs={jobs:2d}   {elapsed:5.2f}s")
    return result

def shuffles(dataframe):
    """Exchange nodes in the physical plan -- one per shuffle."""
    return dataframe._jdf.queryExecution().executedPlan().toString().count("Exchange")

## 1. One `agg`, several answers

The course notebook's own top-$k$ question first, exactly as the chapter reproduces it.

In [3]:
# Find the top 5 fare categories with the highest number of trips
df.groupBy('fare_category').count() \
  .orderBy(['count'], ascending=False).limit(5).show()

+-------------+------+
|fare_category| count|
+-------------+------+
|            6|220980|
|            7|208029|
|            5|206689|
|            8|185057|
|            9|155255|
+-------------+------+



In [4]:
# Four related aggregates that feed one answer. One agg, one shuffle.
summary = df.groupBy("fare_category").agg(
    F.count("*").alias("trips"),
    F.avg("trip_distance").alias("mean_distance"),
    F.approx_count_distinct("medallion").alias("vehicles"),
    F.expr("percentile_approx(fare_amount, 0.5)").alias("median_fare"))

summary.orderBy(F.desc("trips")).limit(5).show()
print("Exchange nodes in that plan:", shuffles(summary))

+-------------+------+------------------+--------+-----------+
|fare_category| trips|     mean_distance|vehicles|median_fare|
+-------------+------+------------------+--------+-----------+
|            6|220980|1.0972071228165452|    7477|        6.0|
|            7|208029| 1.356663926664071|    7185|        7.0|
|            5|206689| 0.844542089806425|    7455|        5.5|
|            8|185057|1.6324369248393744|    7265|        8.0|
|            9|155255|1.9117415864223364|    7280|        9.0|
+-------------+------+------------------+--------+-----------+

Exchange nodes in that plan: 1


In [5]:
# The same four numbers, asked separately.
def one_agg():
    return df.groupBy("fare_category").agg(
        F.count("*"), F.avg("trip_distance"),
        F.approx_count_distinct("medallion"),
        F.expr("percentile_approx(fare_amount, 0.5)")).collect()

def four_queries():
    a = df.groupBy("fare_category").agg(F.count("*")).collect()
    b = df.groupBy("fare_category").agg(F.avg("trip_distance")).collect()
    c = df.groupBy("fare_category").agg(F.approx_count_distinct("medallion")).collect()
    d = df.groupBy("fare_category").agg(F.expr("percentile_approx(fare_amount,0.5)")).collect()
    return a, b, c, d

print("the same four numbers, two ways:")
measure("one agg with four aggregates", one_agg)
measure("four separate queries", four_queries)

separate = [df.groupBy("fare_category").agg(agg)
            for agg in [F.count("*"), F.avg("trip_distance"),
                        F.approx_count_distinct("medallion"),
                        F.expr("percentile_approx(fare_amount,0.5)")]]
print(f"\nExchanges: 1 for the combined query, "
      f"{sum(shuffles(q) for q in separate)} for the four separate ones")

the same four numbers, two ways:


  one agg with four aggregates                 jobs= 2    0.50s


  four separate queries                        jobs= 8    1.16s

Exchanges: 1 for the combined query, 4 for the four separate ones


Nothing in the *results* reveals the difference — the numbers are identical — which is why this
is a habit rather than a discovery. Related aggregates that feed one answer belong in one
`agg`, each aliased so the output schema is deliberate.

### Choosing the counting function knowingly

In [6]:
print("distinct medallions, two ways:")
approx = measure("approx_count_distinct(medallion)",
                 lambda: df.agg(F.approx_count_distinct("medallion")).first()[0])
exact  = measure("count_distinct(medallion)",
                 lambda: df.agg(F.count_distinct("medallion")).first()[0])
print(f"\n  approximate: {approx:,}")
print(f"  exact      : {exact:,}")
print(f"  relative error: {abs(approx - exact) / exact * 100:.2f}% "
      "(the documented default standard deviation is 5%)")

distinct medallions, two ways:
  approx_count_distinct(medallion)             jobs= 2    0.13s


  count_distinct(medallion)                    jobs= 3    0.55s

  approximate: 10,682
  exact      : 10,867
  relative error: 1.70% (the documented default standard deviation is 5%)


An exact distinct count must remember every distinct value it has seen; the approximation keeps
a fixed-size sketch. For a dashboard-grade question about how many vehicles were on the road,
the estimate is the better bargain — and the error above is well inside the bound the
documentation promises.

### Two aggregates that deserve suspicion on sight

In [7]:
# collect_list gathers a whole group into one executor's memory.
small_groups = (df.where(F.col("fare_category") > 150)       # a deliberately tiny slice
                .groupBy("fare_category")
                .agg(F.collect_list("payment_type").alias("payments"),
                     F.count("*").alias("n")))
small_groups.orderBy("fare_category").show(5, truncate=60)

biggest = df.groupBy("payment_type").count().orderBy(F.desc("count")).first()
print(f"the largest group in this table is '{biggest['payment_type']}' with "
      f"{biggest['count']:,} rows.")
print("collect_list over THAT group would build a one-million-element array inside a single")
print("JVM process. Safe for small bounded groups; a reliable source of memory failures")
print("otherwise -- which is why the size of the group, not the size of the table, is the")
print("number to check before writing it.")

+-------------+-------------------------+---+
|fare_category|                 payments|  n|
+-------------+-------------------------+---+
|          152|[CRD, CRD, CRD, CRD, CSH]|  5|
|          153|                    [CRD]|  1|
|          154|                    [CSH]|  1|
|          155|          [CRD, CRD, CRD]|  3|
|          156|                    [CRD]|  1|
+-------------+-------------------------+---+
only showing top 5 rows


the largest group in this table is 'CRD' with 1,011,236 rows.
collect_list over THAT group would build a one-million-element array inside a single
JVM process. Safe for small bounded groups; a reliable source of memory failures
otherwise -- which is why the size of the group, not the size of the table, is the
number to check before writing it.


### `pivot` and the argument that is optional but should not be omitted

In [8]:
values = [r["payment_type"] for r in
          df.select("payment_type").distinct().orderBy("payment_type").collect()]
print("payment types in this file:", values, "\n")

measure("pivot WITHOUT the value list",
        lambda: df.groupBy("fare_category").pivot("payment_type")
                  .sum("total_amount").collect())
measure("pivot WITH the value list",
        lambda: df.groupBy("fare_category").pivot("payment_type", values)
                  .sum("total_amount").collect())

(df.where(F.col("fare_category") < 8)
   .groupBy("fare_category").pivot("payment_type", values)
   .agg(F.round(F.sum("total_amount")).cast("long"))
   .orderBy("fare_category").show())

payment types in this file: ['CRD', 'CSH', 'DIS', 'NOC', 'UNK'] 



  pivot WITHOUT the value list                 jobs= 7    0.46s


  pivot WITH the value list                    jobs= 3    0.26s
+-------------+------+------+----+----+----+
|fare_category|   CRD|   CSH| DIS| NOC| UNK|
+-------------+------+------+----+----+----+
|            2| 10481| 16876|NULL|   3|  49|
|            3| 52093|104086|   4|NULL|  80|
|            4|319258|428599|NULL|NULL| 364|
|            5|654314|707914|NULL|NULL| 893|
|            6|869044|829199|NULL|NULL|1204|
|            7|967405|858687|   8|NULL|1467|
+-------------+------+------+----+----+----+



The extra jobs in the first row are the price of the omission: with no list of expected values,
Spark must run an **additional job just to discover the distinct values** before the real
aggregation can be planned. Supplying them removes that pass and, equally valuable, fixes the
output schema — so a stray new spelling in the pivot column widens the table deliberately or
not at all.

### Map-side combining is a property of aggregates the engine can see into

In [9]:
plan = (df.groupBy("payment_type").agg(F.sum("total_amount"))
        ._jdf.queryExecution().executedPlan().toString())
print("partial_sum in the plan:", "partial_sum" in plan)
for line in plan.split("\n"):
    if "HashAggregate" in line:
        print(" ", " ".join(line.split())[:150])

partial_sum in the plan: True
  +- HashAggregate(keys=[payment_type#10], functions=[sum(total_amount#16)], output=[payment_type#10, sum(total_amount)#33558])
  +- HashAggregate(keys=[payment_type#10], functions=[partial_sum(total_amount#16)], output=[payment_type#10, sum#33770])


Two `HashAggregate` nodes with an `Exchange` between them, the first computing `partial_sum`:
each partition reduces its own rows before the shuffle, and the network carries one partial
result per group per partition rather than the raw rows. That efficiency is available because
`sum` is decomposable and the engine knows it. Hand the same aggregation to an opaque Python
function and partial aggregation disappears — the subject of
[4.6](04.06%20UDF%20Performance%20Hierarchy.ipynb).

## 2. Join types: a policy about unmatched rows

A vehicle reference table, with four fifths of the medallions. The fifth that is missing is what
makes the choice of join type visible: those trips are real, and their vehicle is unknown.

In [10]:
medallions = df.select("medallion").distinct()
vehicles = (medallions
            .where(F.pmod(F.hash("medallion"), F.lit(5)) != 0)     # drop one fifth
            .withColumn("make", F.element_at(
                F.array(F.lit("Ford"), F.lit("Toyota"), F.lit("Honda")),
                (F.pmod(F.hash("medallion"), F.lit(3)) + 1).cast("int")))
            .cache())

trips = df.select("medallion", "pickup_datetime", "fare_amount", "total_amount")
print(f"trips     : {trips.count():,}")
print(f"medallions: {medallions.count():,}")
print(f"vehicles  : {vehicles.count():,}  ({vehicles.count() / medallions.count():.0%} covered)")

trips     : 1,999,999


medallions: 10,867


vehicles  : 8,671  (80% covered)


In [11]:
enriched = trips.join(vehicles, "medallion", "left")       # keep every trip
covered  = trips.join(vehicles, "medallion", "left_semi")  # trips whose vehicle is known
orphans  = trips.join(vehicles, "medallion", "left_anti")  # trips whose vehicle is not
inner    = trips.join(vehicles, "medallion", "inner")

n_trips = trips.count()
for label, d in [("inner    ", inner), ("left     ", enriched),
                 ("left_semi", covered), ("left_anti", orphans)]:
    n = d.count()
    print(f"  {label} -> {n:>9,} rows   ({n - n_trips:+,} against the {n_trips:,} trips)"
          f"   columns: {len(d.columns)}")

  inner     -> 1,599,564 rows   (-400,435 against the 1,999,999 trips)   columns: 5
  left      -> 1,999,999 rows   (+0 against the 1,999,999 trips)   columns: 5


  left_semi -> 1,599,564 rows   (-400,435 against the 1,999,999 trips)   columns: 4
  left_anti ->   400,435 rows   (-1,599,564 against the 1,999,999 trips)   columns: 4


The first line is the chapter's sharpest point about joins. An `inner` join used for enrichment
**deleted 400,000 trips** — a fifth of the table — and reported nothing. It is a filter that no
one wrote, no one reviewed, and nothing in the output announces. A `left` join keeps every fact,
holds nulls where the reference is absent, and thereby keeps the incompleteness *visible*:

In [12]:
print("what the left join reports about its own incompleteness:")
enriched.select(
    F.count("*").alias("rows"),
    F.count(F.when(F.col("make").isNull(), 1)).alias("no_reference_row"),
    F.round(F.avg(F.col("make").isNull().cast("int")) * 100, 1).alias("pct")).show()

print("left_anti is the diagnostic: exactly the keys that failed to match.")
orphans.select("medallion").distinct().show(3, truncate=False)

what the left join reports about its own incompleteness:


+-------+----------------+----+
|   rows|no_reference_row| pct|
+-------+----------------+----+
|1999999|          400435|20.0|
+-------+----------------+----+

left_anti is the diagnostic: exactly the keys that failed to match.


+--------------------------------+
|medallion                       |
+--------------------------------+
|A02946A94C960AF041A251269D57C0B4|
|72EAFBA3FB9F0507C671AD713A622FB6|
|496036713FC662D7165AF3CE796FCAAA|
+--------------------------------+
only showing top 3 rows


### Failure 1: row multiplication

A join matches *every pair* of rows with equal keys, so a key appearing $m$ times on one side
and $n$ times on the other contributes $m \times n$ output rows.

In [13]:
vehicles_dirty = vehicles.unionByName(vehicles)     # the reference table loaded twice
sample = trips.limit(100_000)

n_clean = sample.join(vehicles, "medallion", "left").count()
n_dirty = sample.join(vehicles_dirty, "medallion", "left").count()
print(f"100,000 trips joined to a clean reference table     : {n_clean:,}")
print(f"100,000 trips joined to a DUPLICATED reference table: {n_dirty:,}"
      f"   (x{n_dirty / n_clean:.2f})")
print()
print("the two cheap defenses:")
print("  1. deduplicate the reference side BEFORE the join -- see 4.2")
print(f"  2. count rows before and after: {n_clean:,} -> {n_dirty:,} is one line of code")

100,000 trips joined to a clean reference table     : 100,000
100,000 trips joined to a DUPLICATED reference table: 180,429   (x1.80)

the two cheap defenses:
  1. deduplicate the reference side BEFORE the join -- see 4.2
  2. count rows before and after: 100,000 -> 180,429 is one line of code


Note that the factor is not exactly two: the unmatched fifth of the trips contributes one row
each either way, so only the matched rows double. A multiplication that is *not* a round number
is harder to spot by eye, which is the argument for counting rather than looking.

### Failure 2: null keys match nothing, including each other

In [14]:
with_nulls = trips.limit(1000).unionByName(
    spark.createDataFrame([(None, None, 5.0, 6.0)], trips.schema))
vehicles_with_null = vehicles.limit(100).unionByName(
    spark.createDataFrame([(None, "Unknown")], vehicles.schema))

print("a trip with a null medallion, and a reference row with a null medallion:")
print("  inner    :", with_nulls.join(vehicles_with_null, "medallion", "inner")
      .where(F.col("medallion").isNull()).count(), "rows with a null key")
print("  left     :", with_nulls.join(vehicles_with_null, "medallion", "left")
      .where(F.col("medallion").isNull()).count(), "rows with a null key")
print("  left_anti:", with_nulls.join(vehicles_with_null, "medallion", "left_anti")
      .where(F.col("medallion").isNull()).count(), "rows with a null key")
print()
print("null = null is not true, so the two null-keyed rows did not find each other.")
print("The row vanished from the inner join without comment; the left join kept it;")
print("left_anti reported it as unmatched, which is where it belongs.")

a trip with a null medallion, and a reference row with a null medallion:


  inner    : 0 rows with a null key
  left     : 1 rows with a null key


  left_anti: 1 rows with a null key

null = null is not true, so the two null-keyed rows did not find each other.
The row vanished from the inner join without comment; the left join kept it;
left_anti reported it as unmatched, which is where it belongs.


### Failure 3: ambiguous column names

In [15]:
# Passing the key as a string (or a list) makes Spark keep ONE copy for an equality join.
by_name = trips.join(vehicles, "medallion", "left")
print("join on a string key   -> columns:", by_name.columns)

# Writing the condition as an expression over two same-named columns keeps BOTH copies.
by_expression = trips.join(vehicles, trips.medallion == vehicles.medallion, "left")
print("join on an expression  -> columns:", by_expression.columns)

try:
    by_expression.select("medallion").show(1)
except Exception as e:
    print("\nselect('medallion') afterwards ->", type(e).__name__ + ":",
          getattr(e, "getCondition", lambda: "?")())

# The cure when the expression form is needed: alias both sides and qualify.
t, v = trips.alias("t"), vehicles.alias("v")
qualified = (t.join(v, F.col("t.medallion") == F.col("v.medallion"), "left")
             .select(F.col("t.medallion"), F.col("v.make")))
print("\naliased and qualified  -> columns:", qualified.columns)
qualified.show(3, truncate=False)

join on a string key   -> columns: ['medallion', 'pickup_datetime', 'fare_amount', 'total_amount', 'make']
join on an expression  -> columns: ['medallion', 'pickup_datetime', 'fare_amount', 'total_amount', 'medallion', 'make']

select('medallion') afterwards -> AnalysisException: AMBIGUOUS_REFERENCE

aliased and qualified  -> columns: ['medallion', 'make']
+--------------------------------+-----+
|medallion                       |make |
+--------------------------------+-----+
|07290D3599E7A0D62097A346EFCC1FB5|NULL |
|22D70BF00EEB0ADC83BA8177BB861991|Honda|
|0EC22AAF491A8BD91F279350C2B010FD|NULL |
+--------------------------------+-----+
only showing top 3 rows


The ambiguity surprises no one until the first `select("medallion")` afterwards fails, because
the name no longer picks out a single column. Two remedies, in order of preference: pass the key
as a string or list, which is what the enrichment join above does; or alias the two sides at
join time and qualify every reference through the aliases.

## The two halves, in one query

In [16]:
# A left join for enrichment, then one agg that answers four questions about the result.
report = (trips.join(vehicles, "medallion", "left")
          .withColumn("make", F.coalesce("make", F.lit("(unknown)")))
          .groupBy("make")
          .agg(F.count("*").alias("trips"),
               F.approx_count_distinct("medallion").alias("vehicles"),
               F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
               F.round(F.sum("total_amount")).cast("long").alias("revenue"))
          .orderBy(F.desc("trips")))
print("Exchanges:", shuffles(report))
report.show()

Exchanges: 5


+---------+------+--------+--------+-------+
|     make| trips|vehicles|avg_fare|revenue|
+---------+------+--------+--------+-------+
|   Toyota|562742|    3052|   11.85|7942533|
|    Honda|532973|    2838|   11.92|7564857|
|     Ford|503849|    2623|   11.92|7145082|
|(unknown)|400435|    2202|   11.91|5677035|
+---------+------+--------+--------+-------+



In [17]:
df.unpersist(); vehicles.unpersist()
print("done")

done


## Conclusion

* **One `agg` with four aggregates costs one shuffle; four queries cost four**, and the results
  are identical, so nothing but the instrument tells you which one you wrote.
* **`approx_count_distinct` keeps a fixed-size sketch** and came within a fraction of a percent
  of the exact answer here, at a fraction of the cost. Exact distinctness has to remember
  everything.
* **`collect_list` is bounded by one executor's memory**, so the number to check before using it
  is the size of the largest *group*, not the size of the table.
* **Omitting `pivot`'s value list buys an extra job** over the data and an output schema that
  the data decides.
* **Decomposable aggregates are combined map-side**, visible as `partial_sum` in the plan, and
  that is a property of aggregates the engine can see into.
* **A join type is a decision about which rows survive.** For enrichment the default is `left`,
  because an `inner` join used for enrichment silently deleted a fifth of the trips here.
* **`left_anti` is the diagnostic** — exactly the keys that failed to match — and the first query
  to run when a `left` join comes back full of nulls.
* **Three failures account for most join surprises**: multiplication by duplicated keys, null
  keys that match nothing including each other, and duplicated column names that break the next
  `select`.